# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

## 1. Team (Group Y)

| Role | Name | Student ID |
|------|------|------------|
| Lead | Mohamad Sadough| |
| Member | Peyman Farahani| |
| Member | Ahmed Elsaidy| |


## 2. Mission Title & Research Question

**Title:** Credit Card Fraud Detection

**Research question:**

To what extent can machine learning classification models maximize the detection rate (Recall) of fraudulent transactions while minimizing false alarms, utilizing transaction-level telemetry (amount, category, and spatio-temporal data) within a simulated population of 1.8 Million credit card customers transacting with 800 merchants between January 1, 2019, and December 31, 2020?

**Why it matters:**

Credit card fraud represents a multi-billion-dollar drain on the global economy, directly inflating operational costs for financial institutions, reducing revenue for merchants, and ultimately burdening consumers through higher transaction fees and interest rates. Beyond the direct financial impact, fraud erodes systemic trust in digital payment infrastructures and causes significant administrative distress for identity theft victims. Developing highly precise, real-time predictive models is critical to intercepting malicious activities before clearance, safeguarding economic capital, and ensuring a secure, frictionless ecosystem for digital commerce.

## 3. Data

**Source(s):**

* **Dataset Name:** Credit Card Transactions Fraud Detection Dataset
* **Provider/Author:** Kartik Shenoy (`kartik2112` on Kaggle), simulated via Brandon Harris's *Sparkov Data Generation* tool.
* **URL/Access Method:** [https://www.kaggle.com/datasets/kartik2112/fraud-detection/data](https://www.kaggle.com/datasets/kartik2112/fraud-detection/data)
* **License/Terms of Use:** CC0: Public Domain

**Unit of observation:** One row represents a single credit card transaction attempt (either authorized/legitimate or fraudulent) conducted by a customer at a specific merchant.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
| --- | --- | --- | --- |
| `Unnamed: 0` / Index | Numeric (Integer) | Metadata / ID | Row index counter. |
| `trans_date_trans_time` | String / Datetime | Feature | The exact date and timestamp of the transaction attempt. |
| `cc_num` | Numeric (Integer) / Categorical | Feature | Unique identification number of the customer's credit card. |
| `merchant` | String / Categorical | Feature | Name of the merchant where the transaction was initiated. |
| `category` | String / Categorical | Feature | Category type of the merchant/expense (e.g., `grocery_pos`, `gas_transport`, `entertainment`). |
| `amt` | Numeric (Float) | Feature | The financial amount of the transaction in USD. |
| `first` | String | Feature | First name of the credit card holder. |
| `last` | String | Feature | Last name of the credit card holder. |
| `gender` | String / Categorical | Feature | Gender of the credit card holder (`M` or `F`). |
| `street` | String | Feature | Street address of the credit card holder. |
| `city` | String / Categorical | Feature | City of residence of the credit card holder. |
| `state` | String / Categorical | Feature | State of residence of the credit card holder. |
| `zip` | Numeric (Integer) / Categorical | Feature | Zip code of the credit card holder. |
| `lat` | Numeric (Float) | Feature | Latitude coordinate of the cardholder's location. |
| `long` | Numeric (Float) | Feature | Longitude coordinate of the cardholder's location. |
| `city_pop` | Numeric (Integer) | Feature | Total population size of the cardholder's city. |
| `job` | String / Categorical | Feature | Profession/Occupation of the credit card holder. |
| `dob` | String / Date | Feature | Date of birth of the credit card holder. |
| `trans_num` | String | Feature / ID | A unique alphanumeric identifier string generated for the transaction. |
| `unix_time` | Numeric (Integer) | Feature | The transaction timestamp expressed in Unix epoch format (seconds elapsed). |
| `merch_lat` | Numeric (Float) | Feature | Latitude coordinate of the merchant's location. |
| `merch_long` | Numeric (Float) | Feature | Longitude coordinate of the merchant's location. |
| `is_fraud` | Numeric (Integer / Binary) | Target | Binary indicator flag where `1` represents a fraudulent transaction and `0` represents legitimate behavior. |

**Potential data quality issues:**

* **Simulation Artifacts & Predefined Rules (Selection Bias / Synthetic Limitations):** Because the transactions are synthesized using mathematical profiles (e.g., normal distributions for transaction amounts tailored by specific age/gender demographic brackets), the relationship patterns are deterministic. Predictive models may optimize on arbitrary boundaries set by the simulator's configurations rather than real-world human behavior.
* **Extreme Class Imbalance:** Fraud cases account for less than 0.6% of the overall dataset. While realistic to true fraud operations, models trained naively will struggle with accuracy paradoxes, necessitating stratifications, weighted loss functions, or alternative metrics (like PR-AUC or Recall over basic Accuracy).
* **Data Leakage risk via ID fields:** High cardinality attributes such as `trans_num`, `first`, `last`, and `cc_num` are unique keys that should be dropped or structurally anonymized before model fitting; otherwise, algorithms might simply "memorize" specific fraud-prone identities instead of learning broader fraud patterns.
* **Simplistic Spatial Features:** Merchant coordinates (`merch_lat`, `merch_long`) are mathematically scattered in relation to the cardholder's coordinates (`lat`, `long`). Real-world card-not-present fraud can occur instantly across the globe, whereas this simulation structure makes distance calculations an overly predictive anomaly indicator.
* **Missing Values:** The dataset contains zero missing or null values.

In [15]:
# Data loading & first inspection
# ────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import kagglehub

# 1. Dataset from Kaggle
path = kagglehub.dataset_download("kartik2112/fraud-detection", output_dir="./data")
print("Path to dataset files:", path)

# 2. Load the train and test datasets
data1 = pd.read_csv('./data/fraudTrain.csv')
data2 = pd.read_csv('./data/fraudTest.csv')

# 3. Combine into a single dataframe for comprehensive analysis and preprocessing
df = pd.concat([data1, data2], ignore_index=True)

# 4. First inspection
print("--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
data1.info()
data2.info()
df.info()

print("\n--- Statistical Summary ---")
display(df.describe())

Path to dataset files: ./data
--- First 5 Rows ---


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0



--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  str    
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  str    
 4   category               1296675 non-null  str    
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  str    
 7   last                   1296675 non-null  str    
 8   gender                 1296675 non-null  str    
 9   street                 1296675 non-null  str    
 10  city                   1296675 non-null  str    
 11  state                  1296675 non-null  str    
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long   

,Unnamed: 0,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06,1.852394e+06
mean,5.371934e+05,4.173860e+17,7.006357e+01,4.881326e+04,3.853931e+01,-9.022783e+01,8.864367e+04,1.358674e+09,3.853898e+01,-9.022794e+01,5.210015e-03
std,3.669110e+05,1.309115e+18,1.592540e+02,2.688185e+04,5.071470e+00,1.374789e+01,3.014876e+05,1.819508e+07,5.105604e+00,1.375969e+01,7.199217e-02
min,0.000000e+00,6.041621e+10,1.000000e+00,1.257000e+03,2.002710e+01,-1.656723e+02,2.300000e+01,1.325376e+09,1.902742e+01,-1.666716e+02,0.000000e+00
25%,2.315490e+05,1.800429e+14,9.640000e+00,2.623700e+04,3.466890e+01,-9.679800e+01,7.410000e+02,1.343017e+09,3.474012e+01,-9.689944e+01,0.000000e+00
50%,4.630980e+05,3.521417e+15,4.745000e+01,4.817400e+04,3.935430e+01,-8.747690e+01,2.443000e+03,1.357089e+09,3.936890e+01,-8.744069e+01,0.000000e+00
75%,8.335758e+05,4.642255e+15,8.310000e+01,7.204200e+04,4.194040e+01,-8.015800e+01,2.032800e+04,1.374581e+09,4.195626e+01,-8.024511e+01,0.000000e+00
max,1.296674e+06,4.992346e+18,2.894890e+04,9.992100e+04,6.669330e+01,-6.795030e+01,2.906700e+06,1.388534e+09,6.751027e+01,-6.695090e+01,1.000000e+00


## 4. Planned Methods

### 4a. Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [X] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:*

In fraud detection, it is easy for models to pick up on fake correlations. We will use a Causal Graph (DAG) to formalize our assumptions about the data generation process (e.g., does the transaction category independently drive fraud risk, or is it heavily confounded by user demographics like `age` and `city_pop`?). By using Backdoor adjustment, we can condition on these confounders to isolate the true causal effect of specific transaction behaviors on the likelihood of fraud.

### 4b. Supervised Learning
- [ ] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [X] Other: AdaBoost, Naive Bayes

*Justification:*

We chose Random Forest because it excels with non-linear, tabular data and provides excellent feature importance tracking. AdaBoost is included because boosting algorithms are incredibly effective at focusing on the "hard-to-predict" instances, which is ideal for the minority class (fraud). Naive Bayes will act as our fast, probabilistic baseline.


### 4c. Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [X] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:*

These models will be deployed for transaction profiling and anomaly discovery without relying on the `is_fraud` labels. K-Means clustering will group transactions to see if high-risk micro-clusters emerge naturally based purely on numerical relationships (e.g., specific `amt` thresholds combined with time of day). Hierarchical clustering (applied to a down-sampled subset due to its $O(n^3)$ complexity) will help us visualize the taxonomy of transaction types via a dendrogram, revealing the hierarchical relationships between normal and abnormal spending behaviors.

## 5. Evaluation Strategy

- **Metrics for Supervised Models:** Because the dataset is highly imbalanced, accuracy is a deceptive metric. We will rely on **Precision** (minimizing false positives/declined valid cards), **Recall** (catching actual fraud), **F1 Score**, and **AUROC**. Most importantly, we will use **MCC** (Matthews Correlation Coefficient), which is arguably the most reliable single-value metric for severely imbalanced binary classifications, as it requires the model to perform well on both the negative and positive classes to achieve a high score.

- **Validation of Causal Claims:** To ensure our causal graphs are robust, we will execute **Refutation Tests** using the DoWhy library. Specifically, we will use the *Random Common Cause* test (injecting a random dummy confounder to ensure our estimated causal effect doesn't wildly change) and the *Placebo Treatment* test (replacing the treatment variable with random noise, which should logically drop the causal effect to zero).

- **Baselines:** We will compare our final boosted/ensemble models against a simple Naive Bayes classifier trained on unscaled, imbalanced data to quantify the exact uplift provided by SMOTEENN and our chosen algorithms.


## 6. Work Plan

| Step | Owner | Description |
| --- | --- | --- |
| 1 | Mohamad Sadough | **Data collection & cleaning:** Loading Kaggle data, datetime conversion, categorical encoding (OHE for category), and binary mapping (Gender, Day/Night). |
| 2 | Peyman Farahani | **EDA & Distribution Checks:** Statistical testing for data distributions and visual tools (QQ plots, KDEs) to understand the shape of `amt` and `city_pop`. |
| 3 | Ahmed Elsaidy | **Causal inference block:** Building the DoWhy DAG, identifying confounders, estimating causal effects, and running Refutation/Placebo tests. |
| 4 | Peyman Farahani | **Supervised learning block:** Applying Robust Scaling and SMOTEENN. Training and tuning Random Forest, AdaBoost, and Naive Bayes. Evaluating via MCC, F1, and AUROC. |
| 5 | Mohamad Sadough | **Unsupervised / generative block:** Executing K-Means on the full set to find feature clusters, and running Hierarchical clustering on a sampled subset for taxonomy visualization. |
| 6 | Ahmed Elsaidy | **Synthesis & write-up:** Compiling feature importances, causal findings, and model metrics into a final executive summary. |
